# 🎓 MPCIM Thesis - COMPLETE Analysis

**All Models + Perfect Visualizations + Best Model Selection**

Author: Deni Sulaeman | November 2025

## 1. Setup

In [ ]:
# Install packages (uncomment if needed)
# !pip install pandas numpy matplotlib seaborn plotly scikit-learn xgboost shap imbalanced-learn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, auc
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import shap
import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)
print("✅ Libraries imported!")

## 2. Load Data

Upload your CSV file or specify the path to your data file.

In [ ]:
# Option 1: Use existing processed data
data_file = "../data/processed/full_dataset_processed.csv"

# Option 2: Specify your own data file path
# data_file = "../data/processed/your_data_file.csv"

# Option 3: For Google Colab users (uncomment below)
# from google.colab import files
# uploaded = files.upload()
# data_file = list(uploaded.keys())[0]

print(f"✅ Data file: {data_file}")

## 3. Load & Explore

In [ ]:
df = pd.read_csv(data_file)
print(f"Shape: {df.shape}")
display(df.head())
display(df.describe())
if "has_promotion" in df.columns:
    print(f"\nTarget: {df['has_promotion'].value_counts()}")

## 4. Data Preparation

In [ ]:
y = df["has_promotion"]
X = df.drop(columns=["has_promotion", "employee_id_hash", "employee_id", "employee_name"], errors="ignore")
X = X.select_dtypes(include=[np.number])
if X.isnull().sum().sum() > 0:
    X = X.fillna(X.median())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)
print(f"✅ Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

## 5. Train ALL Models (6 Models)

In [ ]:
# Initialize all models
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss"),
    "SVM": SVC(probability=True, random_state=42),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(100, 50), random_state=42, max_iter=500)
}

results = {}
trained_models = {}

print("🤖 Training 6 Models...")
print("=" * 60)

for name, model in models.items():
    print(f"\nTraining: {name}")
    model.fit(X_train_scaled, y_train_bal)
    trained_models[name] = model
    
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_pred_proba)
    }
    
    print(f"  Accuracy: {results[name]['Accuracy']:.4f}")
    print(f"  F1-Score: {results[name]['F1-Score']:.4f}")
    print(f"  ROC-AUC: {results[name]['ROC-AUC']:.4f}")

print("\n✅ All 6 models trained!")

## 6. Model Comparison

In [ ]:
# Results table
results_df = pd.DataFrame(results).T
print("\n📊 Model Comparison:")
display(results_df.style.highlight_max(axis=0, color="lightgreen"))

# Bar plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
results_df.plot(kind="bar", ax=axes[0], rot=45)
axes[0].set_title("Model Performance Comparison", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Score")
axes[0].legend(loc="lower right")
axes[0].grid(True, alpha=0.3)

# Heatmap
sns.heatmap(results_df.T, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[1])
axes[1].set_title("Model Metrics Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Confusion Matrices (All Models)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                xticklabels=["Not Promoted", "Promoted"],
                yticklabels=["Not Promoted", "Promoted"])
    axes[idx].set_title(f"{name}\nConfusion Matrix", fontweight="bold")
    axes[idx].set_ylabel("Actual")
    axes[idx].set_xlabel("Predicted")

plt.tight_layout()
plt.show()

## 8. ROC Curves (All Models)

In [ ]:
plt.figure(figsize=(10, 8))

for name, model in trained_models.items():
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=2, label="Random")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves - All Models", fontsize=14, fontweight="bold")
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()

## 9. Feature Importance (Tree Models)

In [ ]:
tree_models = ["Random Forest", "Gradient Boosting", "XGBoost"]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, name in enumerate(tree_models):
    model = trained_models[name]
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:10]
    
    axes[idx].barh(range(10), importances[indices], color="skyblue")
    axes[idx].set_yticks(range(10))
    axes[idx].set_yticklabels([X.columns[i] for i in indices], fontsize=9)
    axes[idx].set_xlabel("Importance")
    axes[idx].set_title(f"{name}\nTop 10 Features", fontweight="bold")
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

## 10. SHAP Analysis (XGBoost)

In [ ]:
print("🔍 SHAP Analysis for XGBoost")
print("=" * 60)

best_model = trained_models["XGBoost"]
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_scaled)

print("\n✅ SHAP values calculated")

# Summary plot
print("\n📊 SHAP Summary Plot:")
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns, show=False)
plt.tight_layout()
plt.show()

# Feature importance
print("\n📊 SHAP Feature Importance:")
shap.summary_plot(shap_values, X_test_scaled, feature_names=X.columns, plot_type="bar", show=False)
plt.tight_layout()
plt.show()

## 11. BEST MODEL SELECTION

In [ ]:
# Find best model
print("🏆 BEST MODEL SELECTION")
print("=" * 60)

# Rank by each metric
metrics = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
rankings = {}

for metric in metrics:
    sorted_models = results_df[metric].sort_values(ascending=False)
    print(f"\n📊 Best {metric}:")
    for i, (model, score) in enumerate(sorted_models.head(3).items(), 1):
        print(f"   {i}. {model}: {score:.4f}")
        if model not in rankings:
            rankings[model] = 0
        rankings[model] += (4 - i)  # Points: 3, 2, 1

# Overall best
print("\n" + "=" * 60)
print("🏆 OVERALL RANKING (by total points):")
print("=" * 60)

sorted_rankings = sorted(rankings.items(), key=lambda x: x[1], reverse=True)
for i, (model, points) in enumerate(sorted_rankings, 1):
    print(f"{i}. {model}: {points} points")

best_model_name = sorted_rankings[0][0]
print(f"\n✨ BEST MODEL: {best_model_name} ✨")
print("\nPerformance:")
for metric, value in results[best_model_name].items():
    print(f"  {metric}: {value:.4f}")

## 12. Final Summary

✅ **Analysis Complete!**

### What We Did:
1. Trained 6 ML models
2. Compared all metrics
3. Visualized performance
4. Selected best model

### Next Steps:
1. Use best model for predictions
2. Deploy to production
3. Monitor performance

**Thank you!** 🎓✨